# Análise de Componentes Principais (PCA)

**Objetivo:** reduzir a dimensionalidade do conjunto **breast cancer** (30 variáveis) com PCA, ler o *scree plot* e a variância acumulada, e projetar em 2D — vendo as classes se separarem sem terem sido usadas.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

dados = load_breast_cancer()
X = StandardScaler().fit_transform(dados.data)   # padronizar antes da PCA
y = dados.target
print("X:", X.shape, "-> vamos comprimir 30 variaveis")

## 1. Variância explicada (scree plot)

Cada componente captura uma fração da variância total (seu autovalor normalizado). O *scree plot* mostra quanto cada uma guarda; a curva acumulada mostra quantas bastam.

In [ ]:
pca = PCA().fit(X)
variancia = pca.explained_variance_ratio_
acumulada = np.cumsum(variancia)
for i in range(6):
    print("PC", i+1, "-> variancia", round(variancia[i], 3), "| acumulada", round(acumulada[i], 3))

from plotly.subplots import make_subplots
figura = make_subplots(rows=1, cols=2, subplot_titles=("Variancia por componente", "Variancia acumulada"))
figura.add_trace(go.Bar(x=list(range(1, 11)), y=variancia[:10], marker_color=AZUL), row=1, col=1)
figura.add_trace(go.Scatter(x=list(range(1, 11)), y=acumulada[:10], mode="lines+markers",
                            line=dict(color=VERDE)), row=1, col=2)
figura.add_hline(y=0.9, line_dash="dash", line_color=VERMELHO, row=1, col=2)
figura.update_layout(height=340, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
n90 = int(np.argmax(acumulada >= 0.9)) + 1
print("componentes para reter 90% da variancia:", n90, "de 30")

## 2. Os dados em 2D

Projetando nas duas primeiras componentes, os 30 números viram 2 — e as classes (que a PCA **não** viu) já aparecem quase separadas.

In [ ]:
coords = PCA(n_components=2).fit_transform(X)
figura = go.Figure()
for classe, nome, cor in [(0, "maligno", VERMELHO), (1, "benigno", AZUL)]:
    m = y == classe
    figura.add_trace(go.Scatter(x=coords[m, 0], y=coords[m, 1], mode="markers",
                                marker=dict(color=cor, size=6, opacity=0.6), name=nome))
figura.update_layout(title="breast cancer projetado em 2 componentes principais",
                     xaxis_title="PC1", yaxis_title="PC2", height=420,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. O que carrega cada componente

As **cargas** dizem o peso de cada variável original em cada componente. As cinco maiores da PC1 revelam o que domina a maior direção de variação.

In [ ]:
pca2 = PCA(n_components=2).fit(X)
cargas_pc1 = pca2.components_[0]
ordem = np.argsort(np.abs(cargas_pc1))[::-1][:5]
print("variaveis com maior peso na PC1:")
for j in ordem:
    print("  ", dados.feature_names[j].ljust(24), round(cargas_pc1[j], 3))

## Exercício

Se a PC1 sozinha explica cerca de 44% da variância e a PC2 cerca de 19%, quanto se perde ao olhar só o plano PC1–PC2? Isso invalida a visualização?

<details><summary>Ver resposta</summary>

As duas juntas retêm $\approx 44\% + 19\% = 63\%$; a projeção 2D perde os $\approx 37\%$ restantes, espalhados pelas outras 28 componentes. Não invalida a visualização: 63% da variação num único plano já basta para ver a separação dominante entre maligno e benigno. A ressalva é lembrar que pontos próximos no plano podem diferir nas dimensões descartadas.

</details>